# Stochastic BML — Animations

Three watchable animations illustrating the stochastic BML model
($p_{\text{rand}} = 0.3$ unless stated).

| Animation | What it shows |
|-----------|---------------|
| **1. Three density regimes** | Free flow · Critical · Gridlock — all with $p_{\text{rand}}=0.3$ |
| **2. $p_{\text{rand}}$ sweep** | Fixed critical density; $p_{\text{rand}}$ increases frame by frame — watch stripes dissolve |
| **3. Det vs Stochastic space-time** | Row occupancy building up over time; stochastic near-gridlock shows intermittent gaps |

Run **Kernel → Restart & Run All** to generate all three widgets.
Each animation has play/pause controls.

In [ ]:
# ── Imports, globals, and model classes ───────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
from matplotlib.patches import Patch
from IPython.display import HTML

L_ANIM   = 80     # grid size for animations (smaller = faster)
T_WARMUP = 200    # warmup steps before recording
P_RAND   = 0.3    # default dawdling probability
RHO_C    = 0.32   # deterministic BML critical density (L=80 approx)

plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})


# ── BML base class ─────────────────────────────────────────────────────────
class BML:
    def __init__(self, L, density, r_horiz=0.5, seed=None):
        self.L = L
        self.t = 0
        rng    = np.random.default_rng(seed)
        n_total = int(round(density * L * L))
        n_horiz = int(round(r_horiz * n_total))
        n_vert  = n_total - n_horiz
        flat      = rng.permutation(L * L)
        self.grid = np.zeros(L * L, dtype=np.int8)
        self.grid[flat[:n_horiz]]                 = 1
        self.grid[flat[n_horiz:n_horiz + n_vert]] = 2
        self.grid = self.grid.reshape(L, L)

    def _move_horizontal(self):
        can_move = (self.grid == 1) & (np.roll(self.grid, -1, axis=1) == 0)
        n_moved  = int(can_move.sum())
        if n_moved:
            g = self.grid.copy()
            g[can_move] = 0
            g[np.roll(can_move, 1, axis=1)] = 1
            self.grid = g
        return n_moved

    def _move_vertical(self):
        can_move = (self.grid == 2) & (np.roll(self.grid, 1, axis=0) == 0)
        n_moved  = int(can_move.sum())
        if n_moved:
            g = self.grid.copy()
            g[can_move] = 0
            g[np.roll(can_move, -1, axis=0)] = 2
            self.grid = g
        return n_moved

    def step(self):
        n_total = int((self.grid > 0).sum())
        self._move_horizontal()
        self._move_vertical()
        self.t += 1

    def warmup(self, T):
        for _ in range(T):
            self.step()


# ── StochasticBML ──────────────────────────────────────────────────────────
class StochasticBML(BML):
    def __init__(self, L, density, p_rand=0.0, r_horiz=0.5, seed=None):
        super().__init__(L=L, density=density, r_horiz=r_horiz, seed=seed)
        self.p_rand = p_rand
        self.rng    = np.random.default_rng(seed)

    def _move_horizontal(self):
        can_move = (self.grid == 1) & (np.roll(self.grid, -1, axis=1) == 0)
        if self.p_rand > 0:
            can_move &= ~(self.rng.random(self.grid.shape) < self.p_rand)
        n_moved = int(can_move.sum())
        if n_moved:
            g = self.grid.copy()
            g[can_move] = 0
            g[np.roll(can_move, 1, axis=1)] = 1
            self.grid = g
        return n_moved

    def _move_vertical(self):
        can_move = (self.grid == 2) & (np.roll(self.grid, 1, axis=0) == 0)
        if self.p_rand > 0:
            can_move &= ~(self.rng.random(self.grid.shape) < self.p_rand)
        n_moved = int(can_move.sum())
        if n_moved:
            g = self.grid.copy()
            g[can_move] = 0
            g[np.roll(can_move, -1, axis=0)] = 2
            self.grid = g
        return n_moved


# ── Shared colormap ────────────────────────────────────────────────────────
cmap_bml = mcolors.ListedColormap(['white', 'royalblue', 'tomato'])
norm_bml = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap_bml.N)
print('Setup complete.')

## Animation 1 — Three Density Regimes  ($p_{\text{rand}} = 0.3$)

Three $80 \times 80$ grids run in parallel after a 200-step warmup:

| Panel | Density | Expected behaviour |
|-------|---------|--------------------|
| Left | $\rho = 0.10$ | Free flow — sparse, cars move each step |
| Centre | $\rho \approx \rho_c$ | Critical — clusters form and dissolve; self-organised patterns |
| Right | $\rho = 0.50$ | Near-gridlock — dense, but stochastic dawdling allows occasional flickers |

Compare the **right panel** to the deterministic BML animation: the deterministic
model freezes completely above $\rho_c$, while the stochastic model shows intermittent
movement as dawdling randomly creates gaps.

In [ ]:
# ── Animation 1: Three density regimes ────────────────────────────────────
N_FRAMES_1 = 120
INTERVAL_1 = 80   # ms between frames

scenarios_1 = [
    (0.10,   f'Free Flow  (\u03c1 = 0.10)'),
    (RHO_C,  f'Critical  (\u03c1 \u2248 {RHO_C:.2f})'),
    (0.50,   'Dense  (\u03c1 = 0.50)'),
]

sims_1 = [
    StochasticBML(L=L_ANIM, density=rho, p_rand=P_RAND, seed=i)
    for i, (rho, _) in enumerate(scenarios_1)
]
for sim in sims_1:
    sim.warmup(T_WARMUP)

fig1, axes1 = plt.subplots(1, 3, figsize=(13, 4.5))
fig1.suptitle(f'Stochastic BML: Three Density Regimes  ($p_{{\\rm rand}}={P_RAND}$)', fontsize=12)

legend_elems = [
    Patch(facecolor='royalblue', label='Horizontal \u2192'),
    Patch(facecolor='tomato',    label='Vertical \u2191'),
    Patch(facecolor='white', edgecolor='grey', label='Empty'),
]

ims_1 = []
for ax, sim, (rho, title) in zip(axes1, sims_1, scenarios_1):
    im = ax.imshow(sim.grid, cmap=cmap_bml, norm=norm_bml,
                   interpolation='nearest', animated=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ims_1.append(im)

axes1[0].legend(handles=legend_elems, loc='lower right', fontsize=8)
step_text_1 = fig1.text(0.5, 0.01, f'Step {T_WARMUP}',
                         ha='center', fontsize=9, color='grey')
plt.tight_layout(rect=[0, 0.04, 1, 1])

def update1(frame):
    for im, sim in zip(ims_1, sims_1):
        sim.step()
        im.set_data(sim.grid)
    step_text_1.set_text(f'Step {T_WARMUP + frame + 1}')
    return ims_1 + [step_text_1]

ani1 = animation.FuncAnimation(
    fig1, update1, frames=N_FRAMES_1, interval=INTERVAL_1, blit=True)
plt.close(fig1)
HTML(ani1.to_jshtml())

## Animation 2 — $p_{\text{rand}}$ Sweep at Critical Density

Fixed density $\rho = \rho_c \approx 0.32$. Each frame shows a **fresh grid** warmed up
for 300 steps at a different $p_{\text{rand}}$ value, sweeping from $0$ to $0.9$.

| $p_{\text{rand}}$ | Expected grid state |
|-------------------|--------------------|
| $\approx 0$ | Sharp diagonal stripe patterns — deterministic BML at $\rho_c$ |
| $\approx 0.3$ | Stripes blurred; clusters less defined; occasional gaps in dense regions |
| $\approx 0.9$ | Near-uniform mixing — extreme dawdling prevents any car from building momentum |

This is the 2-D analogue of watching NaSch stop-and-go waves weaken as $p_{\text{rand}} \to 0$.

In [ ]:
# ── Animation 2: p_rand sweep ──────────────────────────────────────────────
N_FRAMES_2  = 120
INTERVAL_2  = 120  # ms
T_WARMUP_2  = 300
p_sweep     = np.linspace(0.0, 0.9, N_FRAMES_2)

print(f'Pre-computing {N_FRAMES_2} grids (each {T_WARMUP_2}-step warmup) ...')
frames_sweep2 = []
for p in p_sweep:
    sim = StochasticBML(L=L_ANIM, density=RHO_C, p_rand=p, seed=99)
    sim.warmup(T_WARMUP_2)
    frames_sweep2.append(sim.grid.copy())
print('Done.')

fig2, ax2 = plt.subplots(figsize=(5.5, 5))
im2 = ax2.imshow(frames_sweep2[0], cmap=cmap_bml, norm=norm_bml,
                  interpolation='nearest', animated=True)
title2 = ax2.set_title(f'$p_{{\\rm rand}} = 0.00$   (\u03c1 = {RHO_C:.2f})', fontsize=11)
ax2.set_xlabel('Column')
ax2.set_ylabel('Row')
legend_elems2 = [
    Patch(facecolor='royalblue', label='Horizontal \u2192'),
    Patch(facecolor='tomato',    label='Vertical \u2191'),
    Patch(facecolor='white', edgecolor='grey', label='Empty'),
]
ax2.legend(handles=legend_elems2, loc='lower right', fontsize=8)
plt.tight_layout()

def update2(frame):
    im2.set_data(frames_sweep2[frame])
    ax2.set_title(
        f'$p_{{\\rm rand}} = {p_sweep[frame]:.2f}$   (\u03c1 = {RHO_C:.2f})', fontsize=11)
    return [im2]

ani2 = animation.FuncAnimation(
    fig2, update2, frames=N_FRAMES_2, interval=INTERVAL_2, blit=False)
plt.close(fig2)
HTML(ani2.to_jshtml())

## Animation 3 — Deterministic vs Stochastic Space-Time

The occupancy of row $L/2$ is recorded over 150 steps and drawn line by line.
Each column in the diagram is a cell; each row is a timestep.
**Black** = occupied, **white** = empty.

Two rows of panels, three columns:

| | Free flow ($\rho=0.10$) | Critical ($\rho \approx \rho_c$) | Near-gridlock ($\rho=0.45$) |
|---|---|---|---|
| **Top** ($p=0$) | Sparse tracks | Intermittent clusters | Row freezes solid |
| **Bottom** ($p=0.3$) | Slightly denser tracks | Blurred clusters | **Intermittent gaps** — stochastic escape from gridlock |

The bottom-right panel is the key result: unlike the deterministic model where
gridlock is an absorbing state, stochastic dawdling allows occasional unjamming events.

In [ ]:
# ── Animation 3: Deterministic vs Stochastic space-time ───────────────────
N_FRAMES_3  = 150
INTERVAL_3  = 60   # ms
ROW         = L_ANIM // 2

st_rhos   = [0.10, RHO_C, 0.45]
st_titles = [
    'Free Flow  (\u03c1=0.10)',
    f'Critical  (\u03c1\u2248{RHO_C:.2f})',
    'Near-Gridlock  (\u03c1=0.45)',
]
p_configs = [(0.0, '$p=0$  (det.)'), (P_RAND, f'$p={P_RAND}$  (stoch.)')]

# Pre-record all space-time data
st_data = {}  # key: (p, rho) -> (N_FRAMES_3, L) int8 array
for p, _ in p_configs:
    for rho in st_rhos:
        sim = StochasticBML(L=L_ANIM, density=rho, p_rand=p, seed=7)
        sim.warmup(T_WARMUP)
        frames = np.zeros((N_FRAMES_3, L_ANIM), dtype=np.int8)
        for t in range(N_FRAMES_3):
            sim.step()
            frames[t] = (sim.grid[ROW] > 0).astype(np.int8)
        st_data[(p, rho)] = frames

# Canvas arrays that fill up frame by frame
canvas = {key: np.zeros((N_FRAMES_3, L_ANIM), dtype=np.int8)
          for key in st_data}

fig3, axes3 = plt.subplots(2, 3, figsize=(14, 7))
im3_list = []
for row_idx, (p, p_label) in enumerate(p_configs):
    for col_idx, (rho, title) in enumerate(zip(st_rhos, st_titles)):
        ax = axes3[row_idx, col_idx]
        im = ax.imshow(
            canvas[(p, rho)], cmap='binary',
            interpolation='nearest', aspect='auto',
            vmin=0, vmax=1, animated=True)
        ax.set_title(f'{p_label}\n{title}', fontsize=9)
        ax.set_xlabel('Column')
        ax.set_ylabel('Time (steps)')
        im3_list.append((im, (p, rho)))

fig3.suptitle(
    'Space\u2013Time Diagrams: Occupancy of Row L/2  '
    '(top = deterministic, bottom = stochastic)',
    fontsize=11)
plt.tight_layout()

def update3(frame):
    artists = []
    for im, key in im3_list:
        canvas[key][frame] = st_data[key][frame]
        im.set_data(canvas[key])
        artists.append(im)
    return artists

ani3 = animation.FuncAnimation(
    fig3, update3, frames=N_FRAMES_3, interval=INTERVAL_3, blit=True)
plt.close(fig3)
HTML(ani3.to_jshtml())